In [1]:
# ==========================================
# Cell 1 - Import Required Libraries
# ==========================================

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# ==========================================
# Cell 2 - Load Dataset
# ==========================================

df = pd.read_csv("../data/Nassau_Candy_Distributor.csv")

print("Dataset loaded successfully.")

Dataset loaded successfully.


In [3]:
df.dtypes

Row ID              int64
Order ID              str
Order Date            str
Ship Date             str
Ship Mode             str
Customer ID         int64
Country/Region        str
City                  str
State/Province        str
Postal Code           str
Division              str
Region                str
Product ID            str
Product Name          str
Sales             float64
Units               int64
Gross Profit      float64
Cost              float64
dtype: object

In [4]:
# ==========================================
# Cell 3 - Convert Date Columns
# ==========================================

df["Order Date"] = pd.to_datetime(
    df["Order Date"],
    dayfirst=True,
    errors="coerce"
)

df["Ship Date"] = pd.to_datetime(
    df["Ship Date"],
    dayfirst=True,
    errors="coerce"
)

print("Date columns converted successfully.")

Date columns converted successfully.


In [5]:
df.dtypes

Row ID                     int64
Order ID                     str
Order Date        datetime64[us]
Ship Date         datetime64[us]
Ship Mode                    str
Customer ID                int64
Country/Region               str
City                         str
State/Province               str
Postal Code                  str
Division                     str
Region                       str
Product ID                   str
Product Name                 str
Sales                    float64
Units                      int64
Gross Profit             float64
Cost                     float64
dtype: object

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10194 entries, 0 to 10193
Data columns (total 18 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   Row ID          10194 non-null  int64         
 1   Order ID        10194 non-null  str           
 2   Order Date      10194 non-null  datetime64[us]
 3   Ship Date       10194 non-null  datetime64[us]
 4   Ship Mode       10194 non-null  str           
 5   Customer ID     10194 non-null  int64         
 6   Country/Region  10194 non-null  str           
 7   City            10194 non-null  str           
 8   State/Province  10194 non-null  str           
 9   Postal Code     10194 non-null  str           
 10  Division        10194 non-null  str           
 11  Region          10194 non-null  str           
 12  Product ID      10194 non-null  str           
 13  Product Name    10194 non-null  str           
 14  Sales           10194 non-null  float64       
 15  Units        

In [7]:
#This is one of the main KPIs required by the project. # ==========================================
# Cell 5 - Create Lead Time (from raw dates)
# ==========================================

# NOTE: The raw "Ship Date" values in this dataset are corrupted
# (years jump 2-5 years ahead of Order Date for every single row).
# We compute it here anyway to document the issue, but this column
# is NOT used for any downstream KPI - see the correction below.

df["Raw Lead Time (Unreliable)"] = (
    df["Ship Date"] - df["Order Date"]
).dt.days

df[["Order Date", "Ship Date", "Raw Lead Time (Unreliable)"]].head()

,Order Date,Ship Date,Raw Lead Time (Unreliable)
0,2024-01-03,2026-06-30,909
1,2024-01-04,2026-07-01,909
2,2024-01-04,2026-07-01,909
3,2024-01-04,2026-07-01,909
4,2024-01-05,2026-07-05,912


In [8]:
# ==========================================
# Cell 6 - Validate Raw Lead Time (diagnostic only)
# ==========================================

negative_lead_time = (df["Raw Lead Time (Unreliable)"] < 0).sum()
print(f"Negative Raw Lead Times: {negative_lead_time}")

print("\nRaw Lead Time stats (from corrupted Ship Date):")
print(df["Raw Lead Time (Unreliable)"].describe())

Negative Raw Lead Times: 0

Raw Lead Time stats (from corrupted Ship Date):
count    10194.000000
mean      1320.841868
std        262.444892
min        904.000000
25%       1271.000000
50%       1274.000000
75%       1638.000000
max       1642.000000
Name: Raw Lead Time (Unreliable), dtype: float64


## Data Quality Issue: Ship Date Correction

The raw `Ship Date` column is corrupted — every row shows a Ship Date 2-5 years after Order Date, giving unrealistic lead times of 900-1,600+ days. Even after correcting for a likely wrong year, the offset flattens to a near-constant ~175-185 days with almost no variance, so there is no real per-shipment signal left to recover from this column.

**Fix:** we simulate a realistic `Lead Time (Days)` from `Ship Mode`, using industry-typical ranges (Same Day = 0, First Class = 1-2, Second Class = 2-4, Standard Class = 4-7 days), and rebuild `Corrected Ship Date` from it. The original `Order Date`, `Ship Date`, and `Raw Lead Time (Unreliable)` columns are kept in the cleaned dataset for transparency, but **all downstream KPIs use `Lead Time (Days)` / `Corrected Ship Date`.**

> ⚠️ **Limitation to disclose in the research paper:** since `Lead Time (Days)` is generated from `Ship Mode` alone, any "lead time by route/region" comparison reflects which ship modes are used on which routes, not independently observed routing performance. Treat lead-time KPIs as a secondary, modeled metric — weight Sales / Gross Profit / Volume more heavily in the Route Efficiency Score.

In [9]:
# ==========================================
# Cell 6b - Generate Corrected Lead Time & Ship Date
# ==========================================

np.random.seed(42)  # reproducible

lead_time_range = {
    "Same Day": (0, 0),
    "First Class": (1, 2),
    "Second Class": (2, 4),
    "Standard Class": (4, 7),
}

def generate_lead_time(ship_mode):
    low, high = lead_time_range[ship_mode]
    return np.random.randint(low, high + 1)

df["Lead Time (Days)"] = df["Ship Mode"].apply(generate_lead_time)

df["Corrected Ship Date"] = (
    df["Order Date"] + pd.to_timedelta(df["Lead Time (Days)"], unit="D")
)

print("Negative corrected lead times:", (df['Lead Time (Days)'] < 0).sum())
df[["Ship Mode", "Order Date", "Corrected Ship Date", "Lead Time (Days)"]].head()

Negative corrected lead times: 0


,Ship Mode,Order Date,Corrected Ship Date,Lead Time (Days)
0,Standard Class,2024-01-03,2024-01-09,6
1,Standard Class,2024-01-04,2024-01-11,7
2,Standard Class,2024-01-04,2024-01-08,4
3,Standard Class,2024-01-04,2024-01-10,6
4,Standard Class,2024-01-05,2024-01-11,6


In [10]:
# ==========================================
# Cell 7 - Product to Factory Mapping
# ==========================================

factory_map = {
    # Lot's O' Nuts
    "Wonka Bar - Nutty Crunch Surprise": "Lot's O' Nuts",
    "Wonka Bar - Fudge Mallows": "Lot's O' Nuts",
    "Wonka Bar -Scrumdiddlyumptious": "Lot's O' Nuts",

    # Wicked Choccy's
    "Wonka Bar - Milk Chocolate": "Wicked Choccy's",
    "Wonka Bar - Triple Dazzle Caramel": "Wicked Choccy's",

    # Sugar Shack
    "Laffy Taffy": "Sugar Shack",
    "SweeTARTS": "Sugar Shack",
    "Nerds": "Sugar Shack",
    "Fun Dip": "Sugar Shack",
    "Fizzy Lifting Drinks": "Sugar Shack",

    # Secret Factory
    "Everlasting Gobstopper": "Secret Factory",
    "Lickable Wallpaper": "Secret Factory",
    "Wonka Gum": "Secret Factory",

    # The Other Factory
    "Hair Toffee": "The Other Factory",
    "Kazookles": "The Other Factory"
}

print("Factory mapping created successfully.")

Factory mapping created successfully.

In [11]:
# ==========================================
# Cell 8 - Add Factory Column
# ==========================================

df["Factory"] = df["Product Name"].map(factory_map)

df[["Product Name", "Factory"]].head(10)

,Product Name,Factory
0,Wonka Bar - Milk Chocolate,Wicked Choccy's
1,Wonka Bar - Triple Dazzle Caramel,Wicked Choccy's
2,Wonka Bar - Nutty Crunch Surprise,Lot's O' Nuts
3,Wonka Bar -Scrumdiddlyumptious,Lot's O' Nuts
4,Wonka Bar - Triple Dazzle Caramel,Wicked Choccy's
5,Wonka Bar -Scrumdiddlyumptious,Lot's O' Nuts
6,Wonka Bar - Triple Dazzle Caramel,Wicked Choccy's
7,Wonka Bar - Milk Chocolate,Wicked Choccy's
8,Wonka Bar - Nutty Crunch Surprise,Lot's O' Nuts
9,Wonka Bar - Milk Chocolate,Wicked Choccy's


In [12]:
# ==========================================
# Cell 9 - Validate Factory Mapping
# ==========================================

missing_factories = df["Factory"].isnull().sum()

print(f"Products without Factory Mapping: {missing_factories}")

Products without Factory Mapping: 0


The project defines a route as:

Factory → Customer State

So we'll create a new column that combines those two values.

In [13]:
# ==========================================
# Cell 10 - Create Route State
# ==========================================

df["Route State"] = (
    df["Factory"] + " → " + df["State/Province"]
)

df[["Factory", "State/Province", "Route State"]].head()

,Factory,State/Province,Route State
0,Wicked Choccy's,Texas,Wicked Choccy's → Texas
1,Wicked Choccy's,Illinois,Wicked Choccy's → Illinois
2,Lot's O' Nuts,Illinois,Lot's O' Nuts → Illinois
3,Lot's O' Nuts,Illinois,Lot's O' Nuts → Illinois
4,Wicked Choccy's,Pennsylvania,Wicked Choccy's → Pennsylvania


In [14]:
# ==========================================
# Cell 11 - Create Route Region
# ==========================================

df["Route Region"] = (
    df["Factory"] + " → " + df["Region"]
)

df[["Factory", "Region", "Route Region"]].head()

,Factory,Region,Route Region
0,Wicked Choccy's,Interior,Wicked Choccy's → Interior
1,Wicked Choccy's,Interior,Wicked Choccy's → Interior
2,Lot's O' Nuts,Interior,Lot's O' Nuts → Interior
3,Lot's O' Nuts,Interior,Lot's O' Nuts → Interior
4,Wicked Choccy's,Atlantic,Wicked Choccy's → Atlantic


In [15]:
# ==========================================
# Cell 12 - Create Delay Flag
# ==========================================

# Threshold applied to the corrected Lead Time (Days), not the raw/broken column.
delay_threshold = 5

df["Delayed"] = df["Lead Time (Days)"] > delay_threshold

df[["Ship Mode", "Lead Time (Days)", "Delayed"]].head(10)

,Ship Mode,Lead Time (Days),Delayed
0,Standard Class,6,True
1,Standard Class,7,True
2,Standard Class,4,False
3,Standard Class,6,True
4,Standard Class,6,True
5,Standard Class,7,True
6,Standard Class,4,False
7,First Class,1,False
8,Standard Class,6,True
9,Standard Class,5,False


In [16]:
# ==========================================
# Cell 13 - Delay Distribution
# ==========================================

df["Delayed"].value_counts()

Delayed
False    7130
True     3064
Name: count, dtype: int64

In [17]:
# ==========================================
# Cell 14 - Save Cleaned Dataset
# ==========================================

import os

# Create folder if it doesn't exist
os.makedirs("../data/processed", exist_ok=True)

# Save cleaned dataset
df.to_csv("../data/processed/Nassau_Candy_Cleaned.csv", index=False)

print("✅ Cleaned dataset saved successfully.")

✅ Cleaned dataset saved successfully.


# Observations

- Successfully converted date columns to datetime format.
- Found the raw `Ship Date` column to be corrupted (900-1,600+ day lead times for every row) and documented it in `Raw Lead Time (Unreliable)`.
- Generated a realistic `Lead Time (Days)` and `Corrected Ship Date` from `Ship Mode`, since the raw dates could not be trusted (see limitation note above).
- Mapped all products to their corresponding factories.
- Created Route State and Route Region features.
- Created a Delay Flag using a lead time threshold of 5 days, applied to the corrected lead time.
- Saved the cleaned dataset (with both raw and corrected fields) for further analysis.